# Nhóm 3 train baseline PaySim bản v2
Chỉ chạy sau khi checkpoint sample/full đã PASS. CPU/None; Internet On khi cần cài thư viện; Add Input ealaxi/paysim1.
Đọc lại cùng full snapshot bằng hash/bytes; không cần upload CSV hoặc gắn cả ba raw snapshot.
Mỗi lần chạy ô 01 tạo thư mục riêng. Bộ ứng viên/feature/ngân sách được chốt trước trong cấu hình.
Mặc định RUN_FINAL_TEST=True: chọn bằng validation, lưu khóa model/ngưỡng, rồi mới chấm test.
Nếu đang thay đổi thiết kế, đặt False trước khi chạy. Không điều chỉnh theo test rồi gọi đó là test độc lập.
Chạy lại một split từng xem chỉ kiểm tái lập, không tạo bộ test độc lập mới.


## Ô 01 Cấu hình và thư mục phiên chạy

In [ ]:
from pathlib import Path
from datetime import datetime,timezone
import os,sys,subprocess,json,hashlib,platform,importlib,importlib.util,zipfile
SOURCE_OVERRIDE=''
EXPECTED_SHA256='16910f90577b0d981bf8ff289714510bb89bc71bff7d3f220f024e287e4eea6b'
EXPECTED_BYTES=493534783
RUN_RF=True
RUN_FINAL_TEST=True
ALERT_BUDGET=0.01
LR_MAX_ITER=500
SESSION=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
ROOT=Path('/kaggle/working/nhom3_ml')/('run_'+SESSION)
for part in ['src','logs','results']:(ROOT/part).mkdir(parents=True,exist_ok=True)
RESULTS=ROOT/'results'
print('RUN:',ROOT,'Python:',platform.python_version(),'CPU:',os.cpu_count())


## Ô 02 Kiểm thư viện trước khi đọc dữ liệu
Giữ các thư viện sẵn có của Kaggle; chỉ cài thư viện thiếu, không nâng toàn bộ image. Nếu cài thất bại, kiểm Internet rồi chạy lại bước này.

In [ ]:
modules={'numpy':'numpy','pandas':'pandas','sklearn':'scikit-learn','scipy':'scipy',
 'joblib':'joblib','threadpoolctl':'threadpoolctl','matplotlib':'matplotlib'}
missing=[pkg for module,pkg in modules.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable,'-m','pip','install',*missing],check=True)
    importlib.invalidate_caches()
for module in modules:importlib.import_module(module)
print('DEPENDENCIES PASS')


## Ô 03 Xuất mã nguồn và hàm chạy có log
Mã nguồn được nhúng sẵn, không cần upload script. Log hiển thị dần khi train và được lưu trong logs/.

In [ ]:
FILES={'audit_paysim.py': '"""Read-only PaySim CSV audit. Python 3.10+, standard library only.\n\nDoes not clean, rewrite, sample, train on, or publish the input dataset.\nExact duplicate detection is deliberately reported as not performed.\n"""\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport math\nimport sys\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\n\nFIELDS = [\n    "step", "type", "amount", "nameOrig", "oldbalanceOrg",\n    "newbalanceOrig", "nameDest", "oldbalanceDest", "newbalanceDest",\n    "isFraud", "isFlaggedFraud",\n]\nTYPES = {"CASH_IN", "CASH_OUT", "DEBIT", "PAYMENT", "TRANSFER"}\nBALANCES = ["oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"]\n\n\ndef scan_csv(stream):\n    reader = csv.DictReader(stream)\n    if reader.fieldnames != FIELDS:\n        raise ValueError(\n            f"Expected exact PaySim header/order: {FIELDS}; got: {reader.fieldnames}. "\n            "Check file/version/encoding; do not silently rename columns."\n        )\n    missing = Counter({field: 0 for field in FIELDS})\n    invalid_numeric = Counter()\n    warnings = Counter()\n    essential_errors = Counter()\n    types = Counter()\n    labels = Counter()\n    by_type_label = Counter()\n    by_step = defaultdict(Counter)\n    examples = []\n    rows = 0\n    previous_step = None\n    amount_min = None\n    amount_max = None\n\n    def issue(row_num, field, reason):\n        if len(examples) < 12:\n            examples.append({"data_row": row_num, "field": field, "reason": reason})\n\n    for row_num, row in enumerate(reader, start=1):\n        rows += 1\n        if None in row or any(row.get(field) is None for field in FIELDS):\n            essential_errors["wrong_number_of_fields"] += 1\n            issue(row_num, "row", "wrong_number_of_fields")\n            continue\n        for field in FIELDS:\n            if row[field].strip().lower() in {"", "null", "none", "nan", "na", "n/a"}:\n                missing[field] += 1\n        for field in ["type", "nameOrig", "nameDest"]:\n            if not row[field].strip():\n                essential_errors[f"empty_{field}"] += 1\n                issue(row_num, field, "empty_essential_field")\n        if row["type"] not in TYPES:\n            essential_errors["unknown_type"] += 1\n            issue(row_num, "type", "unknown_or_noncanonical_type")\n        types[row["type"]] += 1\n        for field in ["nameOrig", "nameDest"]:\n            if row[field] != row[field].strip():\n                warnings[f"outer_whitespace_{field}"] += 1\n\n        nums = {}\n        for field in ["step", "amount", *BALANCES, "isFraud", "isFlaggedFraud"]:\n            try:\n                val = float(row[field])\n                if not math.isfinite(val):\n                    raise ValueError("nonfinite")\n                nums[field] = val\n            except (TypeError, ValueError, OverflowError):\n                invalid_numeric[field] += 1\n                issue(row_num, field, "not_a_finite_number")\n                if field in {"step", "amount", "isFraud"}:\n                    essential_errors[f"invalid_{field}"] += 1\n\n        step = nums.get("step")\n        if step is not None and (step < 1 or not step.is_integer()):\n            essential_errors["invalid_step_range_or_fraction"] += 1\n            issue(row_num, "step", "must_be_positive_integer")\n            step = None\n        if step is not None:\n            step = int(step)\n            if previous_step is not None and step < previous_step:\n                warnings["step_decreases_in_file_order"] += 1\n            previous_step = step\n            by_step[step]["rows"] += 1\n\n        amount = nums.get("amount")\n        if amount is not None:\n            amount_min = amount if amount_min is None else min(amount_min, amount)\n            amount_max = amount if amount_max is None else max(amount_max, amount)\n            if amount < 0:\n                essential_errors["negative_amount"] += 1\n                issue(row_num, "amount", "negative_amount_requires_review")\n            elif amount == 0:\n                warnings["zero_amount_keep_pending_business_review"] += 1\n        label = nums.get("isFraud")\n        if label is not None and label not in {0, 1}:\n            essential_errors["nonbinary_isFraud"] += 1\n            issue(row_num, "isFraud", "must_be_zero_or_one")\n        elif label is not None:\n            label = int(label)\n            labels[str(label)] += 1\n            by_type_label[(row["type"], label)] += 1\n            if step is not None:\n                by_step[step][f"label_{label}"] += 1\n        flag = nums.get("isFlaggedFraud")\n        if flag is not None and flag not in {0, 1}:\n            warnings["nonbinary_isFlaggedFraud"] += 1\n        for field in BALANCES:\n            if field in nums and nums[field] == 0:\n                warnings[f"zero_{field}_not_automatically_missing"] += 1\n            if field in nums and nums[field] < 0:\n                warnings[f"negative_{field}_review_only"] += 1\n\n    for field in ["nameOrig", "nameDest"]:\n        if missing[field]:\n            essential_errors[f"missing_token_{field}"] = missing[field]\n    steps = sorted(by_step)\n    splits = {}\n    if len(steps) >= 3:\n        n_train = min(len(steps) - 2, max(1, int(0.6 * len(steps))))\n        n_val = min(len(steps) - n_train - 1, max(1, int(0.2 * len(steps))))\n        ranges = {\n            "train": steps[:n_train],\n            "validation": steps[n_train:n_train + n_val],\n            "test": steps[n_train + n_val:],\n        }\n        for split_name, selected in ranges.items():\n            splits[split_name] = {\n                "min_step": selected[0], "max_step": selected[-1],\n                "observed_steps": len(selected),\n                "rows": sum(by_step[s]["rows"] for s in selected),\n                "fraud": sum(by_step[s]["label_1"] for s in selected),\n                "legitimate": sum(by_step[s]["label_0"] for s in selected),\n            }\n    blockers = bool(essential_errors) or rows == 0 or not splits\n    both_classes = all(x["fraud"] > 0 and x["legitimate"] > 0 for x in splits.values()) if splits else False\n    return {\n        "rows": rows,\n        "columns": FIELDS,\n        "missing_or_null_tokens_by_column": dict(missing),\n        "invalid_numeric_by_column": dict(invalid_numeric),\n        "essential_errors": dict(essential_errors),\n        "warnings_are_not_deletion_rules": dict(warnings),\n        "label_counts": dict(labels),\n        "fraud_rate": labels["1"] / rows if rows else None,\n        "type_counts": dict(types),\n        "type_label_counts": [\n            {"type": t, "label": label, "rows": count}\n            for (t, label), count in sorted(by_type_label.items())\n        ],\n        "step_min": steps[0] if steps else None,\n        "step_max": steps[-1] if steps else None,\n        "number_of_observed_steps": len(steps),\n        "amount_min": amount_min, "amount_max": amount_max,\n        "proposed_split_by_60_20_20_percent_of_observed_hours_not_rows": splits,\n        "both_classes_in_every_proposed_split": both_classes,\n        "sample_issues": examples,\n        "exact_duplicates": "NOT_CHECKED; no rows removed; repeated account IDs are not duplicates",\n        "feature_policy": {\n            "target": "isFraud",\n            "exclude_from_main_predictors": [*BALANCES, "isFlaggedFraud"],\n            "never_use_as_predictors": ["row_number", "event_id", "isFraud"],\n        },\n        "next_action": (\n            "Proceed to chronological baseline; review warnings and record duplicate policy. "\n            "This is a schema/quality audit, not a guarantee of no leakage or simulator artifacts."\n            if not blockers and both_classes else\n            "Review essential errors and/or chronological split before training. Do not auto-clean."\n        ),\n        "input_was_modified": False,\n    }\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("csv_path", type=Path)\n    parser.add_argument("--report", type=Path, help="New JSON path; existing files are never overwritten")\n    args = parser.parse_args()\n    source = args.csv_path.resolve(strict=True)\n    if not source.is_file():\n        parser.error("csv_path must be a file")\n    if args.report and args.report.exists():\n        parser.error("Report exists. Choose a new output filename.")\n    digest = hashlib.sha256()\n    with source.open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    with source.open("r", encoding="utf-8-sig", newline="") as stream:\n        report = scan_csv(stream)\n    report["source"] = str(source)\n    report["bytes"] = source.stat().st_size\n    report["sha256"] = digest.hexdigest()\n    text = json.dumps(report, ensure_ascii=True, indent=2)\n    if args.report:\n        with args.report.open("x", encoding="utf-8") as stream:\n            stream.write(text + "\\n")\n        print(f"Report saved: {args.report.resolve()}")\n    else:\n        print(text)\n\n\nif __name__ == "__main__":\n    try:\n        main()\n    except (OSError, ValueError, csv.Error) as error:\n        print(f"Audit failed: {error}", file=sys.stderr)\n        sys.exit(2)\n', 'baseline_core.py': 'import csv,hashlib,json,time\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nFIELDS = [\n    "step", "type", "amount", "nameOrig", "oldbalanceOrg", "newbalanceOrig",\n    "nameDest", "oldbalanceDest", "newbalanceDest", "isFraud", "isFlaggedFraud",\n]\nDTYPES = {"step": "int16", "type": "category", "nameOrig": str, "nameDest": str,\n          "isFraud": "uint8", "isFlaggedFraud": "uint8"}\nDTYPES.update({c: "float64" for c in ["amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"]})\n\ndef log(message):\n    print(f"[{time.strftime(\'%H:%M:%S\')}] {message}", flush=True)\n\ndef save_json(path, value):\n    with path.open("x", encoding="utf-8") as stream:\n        json.dump(value, stream, ensure_ascii=False, indent=2, allow_nan=False)\n        stream.write("\\n")\n\ndef profile(source, audit, out):\n    if (out / "profile.json").exists():\n        raise FileExistsError("profile.json exists; use a new output directory")\n    hashes = []\n    step_groups = []\n    zero_amount = {"0": 0, "1": 0}\n    log("Profile: hashing original columns in chunks; raw data will not be modified")\n    for part_num, chunk in enumerate(pd.read_csv(source, dtype=DTYPES, chunksize=400_000), 1):\n        hashes.append(pd.util.hash_pandas_object(chunk[FIELDS], index=False).to_numpy())\n        step_groups.append(chunk.groupby("step", observed=True).agg(rows=("isFraud", "size"), fraud=("isFraud", "sum")))\n        for label in (0, 1):\n            zero_amount[str(label)] += int(((chunk["amount"] == 0) & (chunk["isFraud"] == label)).sum())\n        log(f"Profile: chunk {part_num}, rows {sum(len(x) for x in hashes):,}")\n    all_hashes = np.concatenate(hashes)\n    del hashes\n    original_order = all_hashes.copy()\n    all_hashes.sort()\n    duplicate_hashes = np.unique(all_hashes[1:][all_hashes[1:] == all_hashes[:-1]])\n    del all_hashes\n    duplicate_extra = 0\n    duplicate_involved = 0\n    duplicate_by_label = {"0": 0, "1": 0}\n    if duplicate_hashes.size:\n        # Hashes only find candidates. Confirm equality on ALL original columns.\n        candidates = []\n        cursor = 0\n        for chunk in pd.read_csv(source, dtype=DTYPES, chunksize=400_000):\n            mask = np.isin(original_order[cursor:cursor + len(chunk)], duplicate_hashes)\n            if mask.any():\n                candidates.append(chunk.loc[mask, FIELDS])\n            cursor += len(chunk)\n        possible = pd.concat(candidates, ignore_index=True)\n        extra = possible.duplicated(subset=FIELDS, keep="first")\n        involved = possible.duplicated(subset=FIELDS, keep=False)\n        duplicate_extra = int(extra.sum())\n        duplicate_involved = int(involved.sum())\n        for label in (0, 1):\n            duplicate_by_label[str(label)] = int((extra & (possible["isFraud"] == label)).sum())\n    del original_order\n    hourly = pd.concat(step_groups).groupby(level=0).sum().sort_index()\n    hourly["legitimate"] = hourly["rows"] - hourly["fraud"]\n    hourly["fraud_rate"] = hourly["fraud"] / hourly["rows"]\n    cumulative = hourly["rows"].cumsum()\n    total = int(hourly["rows"].sum())\n    if total != audit["rows"]:\n        raise ValueError("Profile row count disagrees with independent audit")\n    train_end = int(cumulative.index[np.searchsorted(cumulative.to_numpy(), 0.6 * total)])\n    val_end = int(cumulative.index[np.searchsorted(cumulative.to_numpy(), 0.8 * total)])\n    split_ranges = {\n        "train": (int(hourly.index.min()), train_end),\n        "validation": (train_end + 1, val_end),\n        "test": (val_end + 1, int(hourly.index.max())),\n    }\n    split_info = {}\n    for name, (start, end) in split_ranges.items():\n        group = hourly.loc[start:end]\n        n = int(group["rows"].sum())\n        fraud = int(group["fraud"].sum())\n        if not 0 < fraud < n:\n            raise ValueError(f"Split {name} lacks one class; redesign before fitting")\n        split_info[name] = {"min_step": start, "max_step": end, "rows": n,\n                            "fraud": fraud, "legitimate": n - fraud, "fraud_rate": fraud / n}\n    result = {\n        "source": str(source), "source_sha256": audit["sha256"], "rows": total,\n        "exact_duplicate_extra_rows": duplicate_extra,\n        "exact_duplicate_rows_involved": duplicate_involved,\n        "exact_duplicate_extra_by_label": duplicate_by_label,\n        "duplicate_method": "64-bit hash candidates; candidate equality verified on all 11 parsed original columns; no rows deleted",\n        "zero_amount_by_label": zero_amount,\n        "split_method": "whole hours; first cumulative-hour boundaries reaching 60% and 80% of rows; labels not used to choose boundaries",\n        "splits": split_info,\n        "raw_modified": False,\n        "cleaning_decision": "No automatic deletion/imputation/clipping. Preserve raw; handle forbidden predictors and class imbalance in model design.",\n    }\n    hourly.to_csv(out / "hourly_profile.csv", index_label="step")\n    save_json(out / "profile.json", result)\n    log(f"Profile done: duplicate extra rows {duplicate_extra}; split ends {train_end}, {val_end}")\n    return result\n\ndef make_features(frame, categories):\n    hour = (frame["step"].to_numpy(dtype=np.int32) - 1) % 24\n    matrix = np.empty((len(frame), 3 + len(categories)), dtype=np.float32)\n    matrix[:, 0] = np.log1p(frame["amount"].to_numpy(dtype=np.float64))\n    matrix[:, 1] = np.sin(2 * np.pi * hour / 24)\n    matrix[:, 2] = np.cos(2 * np.pi * hour / 24)\n    for index, category in enumerate(categories, 3):\n        matrix[:, index] = (frame["type"] == category).to_numpy(dtype=np.float32)\n    return matrix\n\ndef select_threshold(y, score, budget=0.01):\n    # Whole tied-score groups must be included. The budget is enforced exactly.\n    order = np.argsort(-score, kind="stable")\n    ranked = score[order]\n    positives = np.cumsum(y[order], dtype=np.int64)\n    group_ends = np.r_[np.flatnonzero(ranked[:-1] != ranked[1:]), len(ranked) - 1]\n    allowed = group_ends[(group_ends + 1) <= int(np.floor(len(y) * budget))]\n    if not len(allowed):\n        return float(np.nextafter(np.max(score), np.inf))\n    total_positive = y.sum()\n    best = max(allowed, key=lambda i: (positives[i] / total_positive,\n                                      positives[i] / (i + 1), -(i + 1)))\n    return float(ranked[best])\n\ndef metrics(y, score, threshold):\n    from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score\n    predicted = score >= threshold\n    tn, fp, fn, tp = confusion_matrix(y, predicted, labels=[0, 1]).ravel()\n    precision = float(tp / (tp + fp)) if tp + fp else 0.0\n    recall = float(tp / (tp + fn)) if tp + fn else 0.0\n    return {\n        "rows": len(y), "fraud": int(y.sum()), "prevalence": float(y.mean()),\n        "average_precision": float(average_precision_score(y, score)),\n        "roc_auc_secondary": float(roc_auc_score(y, score)),\n        "threshold": float(threshold), "precision": precision, "recall": recall,\n        "f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,\n        "alerts": int(tp + fp), "alert_rate": float((tp + fp) / len(y)),\n        "false_positives_per_1000_legitimate": float(1000 * fp / (fp + tn)),\n        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),\n    }\n', 'ml_pipeline.py': '"""Validation-first PaySim baseline; raw input is read-only."""\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport platform\nimport time\nimport warnings\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom threadpoolctl import threadpool_limits\n\n\ndef json_out(path, value):\n    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding=\'utf-8\')\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef versions():\n    result = {\'python\': platform.python_version()}\n    for name in [\'numpy\', \'pandas\', \'scikit-learn\', \'scipy\', \'joblib\', \'threadpoolctl\', \'matplotlib\']:\n        result[name] = importlib.metadata.version(name)\n    return result\n\n\ndef load_frame(source):\n    return pd.read_csv(source, usecols=[\'step\', \'type\', \'amount\', \'isFraud\', \'isFlaggedFraud\'],\n                       dtype={\'step\': \'int32\', \'type\': \'category\', \'amount\': \'float64\',\n                              \'isFraud\': \'uint8\', \'isFlaggedFraud\': \'uint8\'})\n\n\ndef checked_inputs(source, audit_path, out):\n    audit = json.loads(Path(audit_path).read_text(encoding=\'utf-8\'))\n    if audit[\'essential_errors\'] or audit[\'rows\'] == 0:\n        raise ValueError(\'Audit errors: review source before modeling\')\n    if digest(source) != audit[\'sha256\']:\n        raise ValueError(\'Source differs from audited snapshot\')\n    prof = json.loads((out / \'profile.json\').read_text(encoding=\'utf-8\'))\n    if prof[\'source_sha256\'] != audit[\'sha256\']:\n        raise ValueError(\'Profile belongs to another snapshot\')\n    return audit, prof\n\n\ndef plot_scores(y, score, threshold, prefix, name):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    from sklearn.metrics import PrecisionRecallDisplay, ConfusionMatrixDisplay\n    fig, ax = plt.subplots(figsize=(6, 5))\n    PrecisionRecallDisplay.from_predictions(y, score, ax=ax, name=name)\n    fig.tight_layout(); fig.savefig(str(prefix) + \'_pr_curve.png\', dpi=170); plt.close(fig)\n    fig, ax = plt.subplots(figsize=(5, 5))\n    ConfusionMatrixDisplay.from_predictions(y, score >= threshold, ax=ax, labels=[0, 1],\n        display_labels=[\'Legitimate\', \'Fraud\'], colorbar=False)\n    fig.tight_layout(); fig.savefig(str(prefix) + \'_confusion_matrix.png\', dpi=170); plt.close(fig)\n\n\ndef fit_validation(source, audit_path, out, budget, run_rf, lr_max_iter):\n    from baseline_core import make_features, select_threshold, metrics\n    if (out / \'model_manifest.json\').exists():\n        raise FileExistsError(\'Locked model already exists. Start a new notebook run for a new experiment.\')\n    audit, prof = checked_inputs(source, audit_path, out)\n    frame = load_frame(source)\n    masks = {name: frame.step.between(s[\'min_step\'], s[\'max_step\']).to_numpy()\n             for name, s in prof[\'splits\'].items()}\n    categories = sorted(frame.loc[masks[\'train\'], \'type\'].unique().tolist())\n    tx = make_features(frame.loc[masks[\'train\']], categories)\n    vx = make_features(frame.loc[masks[\'validation\']], categories)\n    ty = frame.loc[masks[\'train\'], \'isFraud\'].to_numpy()\n    vy = frame.loc[masks[\'validation\'], \'isFraud\'].to_numpy()\n    candidates = [(f\'LR_weight_{w}\', make_pipeline(StandardScaler(), LogisticRegression(\n        C=1, class_weight={0: 1., 1: float(w)}, max_iter=lr_max_iter, random_state=42)))\n        for w in (1, 10, 50)]\n    if run_rf:\n        candidates.append((\'RF_50_depth8\', RandomForestClassifier(n_estimators=50, max_depth=8,\n            min_samples_leaf=20, max_samples=0.5, class_weight=\'balanced_subsample\', n_jobs=2, random_state=42)))\n    records, best, best_key = [], None, None\n    with threadpool_limits(limits=2):\n        for name, model in candidates:\n            print(\'FITTING\', name, \'train rows\', len(ty), flush=True)\n            start = time.perf_counter()\n            with warnings.catch_warnings(record=True) as caught:\n                warnings.simplefilter(\'always\'); model.fit(tx, ty)\n            seconds = time.perf_counter() - start\n            score = model.predict_proba(vx)[:, 1]\n            threshold = select_threshold(vy, score, budget=budget)\n            result = metrics(vy, score, threshold)\n            record = dict(model=name, fit_seconds=seconds,\n                warnings=[str(w.message) for w in caught], validation=result)\n            records.append(record)\n            key = (result[\'recall\'], result[\'precision\'], result[\'average_precision\'])\n            if best_key is None or key > best_key:\n                best_key, best = key, (name, model, threshold, score.copy())\n            print(name, result, \'WARNINGS\', record[\'warnings\'], flush=True)\n    name, model, threshold, score = best\n    model_path = out / \'selected_model.joblib\'\n    joblib.dump(model, model_path)\n    restored = joblib.load(model_path)  # this artifact was created in this run\n    np.testing.assert_allclose(restored.predict_proba(vx[:64])[:, 1], score[:64], rtol=1e-7, atol=1e-10)\n    manifest = dict(model=name, threshold=float(threshold), type_categories=categories,\n        feature_order=[\'log1p_amount\', \'hour_sin\', \'hour_cos\'] + [\'type_\' + c for c in categories],\n        feature_version=\'safe_v1_logamount_hour_type\', source_sha256=audit[\'sha256\'],\n        model_sha256=digest(model_path), splits=prof[\'splits\'], versions=versions(),\n        model_library=\'sklearn\', validation_alert_budget=budget,\n        selection_policy=\'maximize validation recall then precision then AP, under alert budget; preserve score ties; no test tuning\',\n        excluded_predictors=[\'isFraud\', \'isFlaggedFraud\', \'nameOrig\', \'nameDest\', \'oldbalanceOrg\',\n                             \'newbalanceOrig\', \'oldbalanceDest\', \'newbalanceDest\', \'event_id\'],\n        warning=\'Synthetic dataset; scores are not calibrated real-world fraud risk. Prior test exposure must be disclosed.\',\n        code_sha256=digest(Path(__file__)), feature_code_sha256=digest(Path(__file__).with_name(\'baseline_core.py\')))\n    plot_scores(vy, score, threshold, out / \'validation\', name)\n    json_out(out / \'model_manifest.json\', manifest)  # model and threshold locked before any test scoring\n    json_out(out / \'validation_results.json\', records)\n    pd.json_normalize(records).to_csv(out / \'validation_metrics.csv\', index=False)\n    json_out(out / \'reload_check.json\', dict(status=\'PASS\', rows=min(64, len(vy)), split=\'validation\'))\n    json_out(out / \'environment.json\', manifest[\'versions\'])\n    (out / \'requirements-lock.txt\').write_text(\'\\n\'.join(f\'{k}=={v}\' for k, v in manifest[\'versions\'].items() if k != \'python\') + \'\\n\')\n    print(\'VALIDATION PASS - MODEL LOCKED - NO TEST SCORED\', flush=True)\n\n\ndef evaluate_test(source, audit_path, out):\n    from baseline_core import make_features, metrics\n    if (out / \'test_metrics.json\').exists():\n        raise FileExistsError(\'Test already scored in this run; do not overwrite it\')\n    audit, prof = checked_inputs(source, audit_path, out)\n    manifest = json.loads((out / \'model_manifest.json\').read_text(encoding=\'utf-8\'))\n    if manifest[\'source_sha256\'] != audit[\'sha256\'] or manifest[\'splits\'] != prof[\'splits\']:\n        raise ValueError(\'Locked source/splits mismatch\')\n    if digest(Path(__file__)) != manifest[\'code_sha256\'] or digest(Path(__file__).with_name(\'baseline_core.py\')) != manifest[\'feature_code_sha256\']:\n        raise ValueError(\'Scoring/feature code changed after model selection\')\n    if digest(out / \'selected_model.joblib\') != manifest[\'model_sha256\']:\n        raise ValueError(\'Model artifact changed after selection\')\n    for name, expected in manifest[\'versions\'].items():\n        current = platform.python_version() if name == \'python\' else importlib.metadata.version(name)\n        if current != expected: raise ValueError(\'Version changed after model selection: \' + name)\n    model = joblib.load(out / \'selected_model.joblib\')\n    frame = load_frame(source);s = prof[\'splits\'][\'test\']\n    frame = frame.loc[frame.step.between(s[\'min_step\'], s[\'max_step\'])].copy()\n    x = make_features(frame, manifest[\'type_categories\']);y = frame.isFraud.to_numpy()\n    start = time.perf_counter();score = model.predict_proba(x)[:, 1];seconds = time.perf_counter() - start\n    threshold = manifest[\'threshold\'];result = metrics(y, score, threshold)\n    fixed_rule = ((frame.type == \'TRANSFER\') & (frame.amount > 200000)).to_numpy(dtype=float)\n    bases = {\'always_legitimate\': metrics(y, np.zeros(len(y)), 0.5),\n             \'transfer_gt_200000\': metrics(y, fixed_rule, 0.5),\n             \'dataset_flag_rule\': metrics(y, frame.isFlaggedFraud.to_numpy(dtype=float), 0.5)}\n    # Render first; a plot failure does not leave a false completion marker.\n    plot_scores(y, score, threshold, out / \'test\', manifest[\'model\'])\n    pd.DataFrame([dict(model=manifest[\'model\'], **result)] +\n                 [dict(model=k, **v) for k, v in bases.items()]).to_csv(out / \'test_metrics.csv\', index=False)\n    demo = frame[[\'step\', \'type\', \'amount\']].head(20).copy()\n    demo[\'score\'] = score[:len(demo)]; demo[\'threshold\'] = threshold\n    demo[\'predicted_fraud\'] = demo.score >= threshold\n    demo.to_csv(out / \'prediction_examples.csv\', index=False)\n    json_out(out / \'test_metrics.json\', dict(selected_model=manifest[\'model\'], metrics=result, fixed_baselines=bases,\n        source_sha256=audit[\'sha256\'], model_sha256=manifest[\'model_sha256\'],\n        offline_bulk_predict_seconds=seconds, test_exposure=\'Re-running a known split is reproducibility, not a new independent test.\',\n        warning=\'Batch predict timing is not Kafka/Spark end-to-end latency or throughput. Validation alert budget may not hold on test.\'))\n    print(\'TEST PASS\', json.dumps(result), flush=True)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--stage\', choices=[\'validation\', \'test\'], required=True)\n    ap.add_argument(\'--csv\', type=Path, required=True)\n    ap.add_argument(\'--audit\', type=Path, required=True)\n    ap.add_argument(\'--out\', type=Path, required=True)\n    ap.add_argument(\'--budget\', type=float, default=0.01)\n    ap.add_argument(\'--skip-rf\', action=\'store_true\')\n    ap.add_argument(\'--lr-max-iter\', type=int, default=500)\n    a = ap.parse_args()\n    if not 0 < a.budget < 1: raise ValueError(\'Budget must be between zero and one\')\n    if a.stage == \'validation\':\n        fit_validation(a.csv, a.audit, a.out, a.budget, not a.skip_rf, a.lr_max_iter)\n    else:\n        evaluate_test(a.csv, a.audit, a.out)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'source_utils.py': '"""Find a specific PaySim snapshot, avoiding samples and duplicate-input ambiguity."""\nimport csv\nimport hashlib\nfrom pathlib import Path\n\nFIELDS = [\'step\', \'type\', \'amount\', \'nameOrig\', \'oldbalanceOrg\', \'newbalanceOrig\',\n          \'nameDest\', \'oldbalanceDest\', \'newbalanceDest\', \'isFraud\', \'isFlaggedFraud\']\n\n\ndef sha256_file(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for b in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef find_source(input_root, override, expected_hash, expected_bytes):\n    if not expected_hash or len(expected_hash) != 64 or expected_bytes <= 0:\n        raise ValueError(\'Set the expected SHA-256 and bytes from checkpoint source metadata\')\n    paths = [Path(override)] if override else sorted(Path(input_root).rglob(\'*.csv\'))\n    candidates = []\n    for p in paths:\n        if not p.is_file() or p.stat().st_size != expected_bytes:\n            continue\n        try:\n            with p.open(encoding=\'utf-8-sig\', newline=\'\') as f:\n                if next(csv.reader(f), None) == FIELDS:\n                    candidates.append(p)\n        except (OSError, UnicodeError, csv.Error):\n            continue\n    for p in candidates:\n        actual = sha256_file(p)\n        if actual == expected_hash:\n            return p, actual\n    raise ValueError(\'No matching FULL PaySim snapshot. Add Input ealaxi/paysim1; compare source metadata. Do not select the 1000-row sample or change expected hash without reviewing the source.\')\n'}
for name,text in FILES.items():(ROOT/'src'/name).write_text(text,encoding='utf-8')
sys.path.insert(0,str(ROOT/'src'))
# Clear prior-run imports if this notebook is rerun in the same kernel.
for module in ['source_utils','baseline_core']:
    sys.modules.pop(module,None)
from source_utils import find_source
import baseline_core as core

def run_job(script,args,log_name):
    command=[sys.executable,str(ROOT/'src'/script),*map(str,args)]
    with (ROOT/'logs'/log_name).open('w',encoding='utf-8') as log:
        log.write('$ '+' '.join(command)+'\n');log.flush()
        proc=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace')
        for line in proc.stdout:
            print(line,end='',flush=True);log.write(line);log.flush()
        rc=proc.wait()
    if rc:raise RuntimeError('Job failed; read '+str(ROOT/'logs'/log_name))


## Ô 04 Xác nhận đúng full snapshot
EXPECTED_SHA256/EXPECTED_BYTES lấy từ bộ kết quả của nhóm. Nếu checkpoint v2 ghi snapshot khác, đối chiếu file Nhom3_Checkpoint_to_Train.json rồi mới cập nhật. Nguồn không khớp sẽ dừng; sample 1000 dòng không được chọn thay full.

In [ ]:
SOURCE,SOURCE_HASH=find_source(Path('/kaggle/input'),SOURCE_OVERRIDE,EXPECTED_SHA256,EXPECTED_BYTES)
source_meta=dict(dataset='paysim',source_url='https://www.kaggle.com/datasets/ealaxi/paysim1',
 path=str(SOURCE),sha256=SOURCE_HASH,bytes=SOURCE.stat().st_size,
 snapshot_check='PASS against expected checkpoint snapshot',accessed_at_utc=datetime.now(timezone.utc).isoformat())
(ROOT/'source.json').write_text(json.dumps(source_meta,indent=2),encoding='utf-8')
print('SOURCE PASS',json.dumps(source_meta,indent=2))


## Ô 05 Audit và chia theo thời gian
Giữ nguyên raw. Audit không xóa amount=0 hoặc coi số dư 0 là missing. Profile kiểm trùng và chia gần 60/20/20 theo số HÀNG, giữ nguyên mỗi giờ. Dùng splits trong profile.json cho ML; split sơ bộ theo số giờ trong audit không phải split train. Bước audit full có thể mất vài phút.

In [ ]:
AUDIT=ROOT/'audit.json'
if not AUDIT.exists():run_job('audit_paysim.py',[SOURCE,'--report',AUDIT],'audit.log')
audit=json.loads(AUDIT.read_text(encoding='utf-8'))
if audit['sha256']!=SOURCE_HASH or audit['essential_errors']:raise ValueError('Audit errors/hash mismatch; review source')
if not (RESULTS/'profile.json').exists():core.profile(SOURCE,audit,RESULTS)
prof=json.loads((RESULTS/'profile.json').read_text(encoding='utf-8'))
if prof['source_sha256']!=SOURCE_HASH:raise ValueError('Profile snapshot mismatch')
print('PROFILE PASS',json.dumps(prof['splits'],indent=2))


## Ô 06 Train và khóa mô hình bằng validation
LR weight 1/10/50; RF 50 cây depth 8. Features: log1p(amount), one-hot type và chu kỳ giờ từ step. Bỏ ID, nhãn, cờ luật, số dư khỏi predictors. Scaler chỉ fit train. Chọn recall cao nhất dưới ngân sách cảnh báo validation 1%, rồi precision và AP; giữ nguyên nhóm đồng điểm. Ngân sách này là thiết kế thí nghiệm, không phải yêu cầu Buổi 6.

In [ ]:
args=['--stage','validation','--csv',SOURCE,'--audit',AUDIT,'--out',RESULTS,
 '--budget',str(ALERT_BUDGET),'--lr-max-iter',str(LR_MAX_ITER)]
if not RUN_RF:args.append('--skip-rf')
run_job('ml_pipeline.py',args,'validation.log')
print('Validation metrics:',RESULTS/'validation_metrics.csv')


## Ô 07 Test sau khi đã khóa
Chấm selected model và các luật cố định; không chọn lại bằng test. AP là average precision, khác PR-AUC tích phân hình thang. Ngưỡng có thể vượt ngân sách 1% ở test khi phân bố thay đổi. Thời gian predict theo lô không phải latency/throughput Kafka-Spark.

In [ ]:
if RUN_FINAL_TEST:
    run_job('ml_pipeline.py',['--stage','test','--csv',SOURCE,'--audit',AUDIT,'--out',RESULTS],'test.log')
else:
    print('TEST SKIPPED by configuration; export contains validation only.')


## Ô 08 Xem kết quả và đóng gói
Save Version → Save & Run All, chờ thành công và tải ZIP từ Output. Model sklearn .joblib cần cùng feature/scaler/category/version khi phục vụ; không load bằng Spark MLlib load. Manifest chứa hash model và version thực tế.

In [ ]:
import pandas as pd
from IPython.display import display,Image
required=['profile.json','validation_metrics.csv','selected_model.joblib','model_manifest.json',
 'reload_check.json','validation_pr_curve.png','validation_confusion_matrix.png']
if RUN_FINAL_TEST:required+=['test_metrics.json','test_metrics.csv','test_pr_curve.png','test_confusion_matrix.png','prediction_examples.csv']
assert all((RESULTS/name).is_file() for name in required),'Missing artifacts; inspect prior cells'
print('VALIDATION');display(pd.read_csv(RESULTS/'validation_metrics.csv'))
if RUN_FINAL_TEST:
    print('TEST');display(pd.read_csv(RESULTS/'test_metrics.csv'))
    display(Image(filename=str(RESULTS/'test_confusion_matrix.png')))
    display(Image(filename=str(RESULTS/'test_pr_curve.png')))
freeze=subprocess.run([sys.executable,'-m','pip','freeze'],capture_output=True,text=True,check=True)
(ROOT/'environment_freeze.txt').write_text(freeze.stdout,encoding='utf-8')
status=dict(status='PASS',session=SESSION,source_sha256=SOURCE_HASH,
 validation='PASS',test='PASS' if RUN_FINAL_TEST else 'SKIPPED',
 model_manifest=str(RESULTS/'model_manifest.json'))
(ROOT/'pipeline_status.json').write_text(json.dumps(status,indent=2),encoding='utf-8')
archive=Path('/kaggle/working/Nhom3_Train_artifacts_v2.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in ROOT.rglob('*'):
        if p.is_file() and '__pycache__' not in p.parts:z.write(p,p.relative_to(ROOT))
print('ML PIPELINE PASS',json.dumps(status,indent=2))
print('DOWNLOAD:',archive)
